IMPORTS

In [7]:
import sys
import os

# Add project root to Python path (CRITICAL FIX)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Now imports will work
from src.data_loader import load_data
from src.preprocessing import remove_duplicates, handle_missing_values, convert_datetime
from src.feature_engineering import create_time_features

print("✅ Imports successful! Project root:", project_root)

✅ Imports successful! Project root: /home/dag-dagne/Desktop/10 Acadamy/week 5/fraud-detection-interim2


In [8]:
# Load datasets
fraud_df = load_data("../data/raw/Fraud_Data.csv")
credit_df = load_data("../data/raw/creditcard.csv")
ip_df = load_data("../data/raw/IpAddress_to_Country.csv")

print("Fraud shape:", fraud_df.shape)
print("IP shape:", ip_df.shape)

Loaded: ../data/raw/Fraud_Data.csv | Shape: (151112, 11)
Loaded: ../data/raw/creditcard.csv | Shape: (284807, 31)
Loaded: ../data/raw/IpAddress_to_Country.csv | Shape: (138846, 3)
Fraud shape: (151112, 11)
IP shape: (138846, 3)


In [9]:
fraud_df = remove_duplicates(fraud_df)
fraud_df = handle_missing_values(fraud_df)
fraud_df = convert_datetime(fraud_df, ["signup_time", "purchase_time"])

In [11]:
fraud_df = create_time_features(fraud_df)


# Drop high-cardinality columns BEFORE encoding
drop_cols = ['user_id', 'device_id', 'ip_address', 'ip_int']
fraud_df = fraud_df.drop(columns=drop_cols, errors='ignore')

print("Remaining columns:", fraud_df.columns.tolist())

Remaining columns: ['signup_time', 'purchase_time', 'purchase_value', 'source', 'browser', 'sex', 'age', 'class', 'hour_of_day', 'day_of_week', 'time_since_signup']


In [12]:
y = fraud_df["class"].copy()
X = fraud_df.drop(columns=["class"]).copy()

In [13]:
for col in X.select_dtypes(include=['datetime64']).columns:
    X[col] = X[col].astype('int64') // 10**9

In [15]:
# CELL 7 — ENCODE CATEGORICAL (Safe version - NO 'country')
cat_cols = ['source', 'browser', 'sex']   

X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print("✅ Final shape after encoding:", X.shape)
print("Memory usage (MB):", round(X.memory_usage(deep=True).sum() / (1024**2), 2))
print("Sample columns:", X.columns.tolist()[:10])  # Preview

✅ Final shape after encoding: (151112, 14)
Memory usage (MB): 7.93
Sample columns: ['signup_time', 'purchase_time', 'purchase_value', 'age', 'hour_of_day', 'day_of_week', 'time_since_signup', 'source_Direct', 'source_SEO', 'browser_FireFox']


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)

Train shape: (120889, 14)


In [17]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_res.value_counts().to_dict())

Before SMOTE: {0: 109568, 1: 11321}
After SMOTE: {0: 109568, 1: 109568}
